# 01 — Exploratory Data Analysis (EDA)

**Vietnam Air Quality — Historical 2021 + WAQI API**

Mục tiêu:
1. Hiểu phân phối AQI theo thành phố, theo thời gian
2. Correlation giữa AQI và các pollutant / weather variables
3. Seasonal decomposition (trend, seasonality)
4. Missing value pattern analysis
5. Geospatial visualization các trạm quan trắc

Dataset:
- `raw_data_csv/historical_air_quality_2021_en.csv` — Kaggle historical data (2021)
- Silver `fact_aqi` Parquet (optional, via Athena/S3)

## 1. Setup & Load Data

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
sns.set_style('whitegrid')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print(f"pandas: {pd.__version__}")
print(f"numpy : {np.__version__}")

In [ ]:
CSV_PATH = Path('../raw_data_csv/historical_air_quality_2021_en.csv')

raw = pd.read_csv(CSV_PATH, na_values=['-', '', '#NAME?'], low_memory=False)
print(f"Shape: {raw.shape}")
raw.head()

In [ ]:
rename_map = {
    'Station ID'        : 'station_id',
    'AQI index'         : 'aqi',
    'Location'          : 'geo',
    'Station name'      : 'station_name',
    'Dominent pollutant': 'dominant_pollutant',
    'CO'                : 'co',
    'Humidity'          : 'humidity',
    'NO2'               : 'no2',
    'O3'                : 'o3',
    'Pressure'          : 'pressure',
    'PM10'              : 'pm10',
    'PM2.5'             : 'pm25',
    'SO2'               : 'so2',
    'Temperature'       : 'temperature',
    'Wind'              : 'wind',
    'Data Time S'       : 'measured_at',
}

df = raw.rename(columns=rename_map).copy()

for col in ['pressure']:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)

num_cols = ['aqi', 'co', 'humidity', 'no2', 'o3', 'pressure',
            'pm10', 'pm25', 'so2', 'temperature', 'wind']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['measured_at'] = pd.to_datetime(df['measured_at'], errors='coerce')

geo = df['geo'].astype(str).str.split(',', expand=True)
df['lat'] = pd.to_numeric(geo[0], errors='coerce')
df['lon'] = pd.to_numeric(geo[1], errors='coerce')

def extract_city(name):
    if pd.isna(name):
        return 'unknown'
    name = str(name).lower()
    if 'hanoi' in name or 'hà nội' in name or 'ha noi' in name:
        return 'ha-noi'
    if 'ho chi minh' in name or 'hồ chí minh' in name:
        return 'ho-chi-minh-city'
    if 'da nang' in name or 'đà nẵng' in name:
        return 'da-nang'
    if 'gia lai' in name or 'pleiku' in name:
        return 'gia-lai'
    if 'cao bằng' in name or 'cao bang' in name:
        return 'cao-bang'
    return 'other'

df['queried_city'] = df['station_name'].apply(extract_city)

df = df[['station_id', 'station_name', 'queried_city', 'lat', 'lon',
        'measured_at', 'aqi', 'dominant_pollutant',
        'pm25', 'pm10', 'co', 'no2', 'o3', 'so2',
        'humidity', 'temperature', 'pressure', 'wind']]

print(f"After cleanup: {df.shape}")
print(f"Date range   : {df['measured_at'].min()} → {df['measured_at'].max()}")
print(f"Cities       : {df['queried_city'].value_counts().to_dict()}")
df.head()

## 2. Data Quality Overview

In [ ]:
print('Missing values (%)')
miss_pct = df.isna().mean().mul(100).round(2).sort_values(ascending=False)
print(miss_pct.to_string())

fig, ax = plt.subplots(figsize=(11, 4))
miss_pct.plot.bar(ax=ax, color='steelblue')
ax.set_title('Missing Values by Column (%)')
ax.set_ylabel('% missing')
ax.axhline(5, color='red', linestyle='--', label='5% threshold')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
print('Numeric summary:')
df[num_cols].describe().T.round(2)

## 3. AQI Distribution by City

In [ ]:
target_cities = ['ha-noi', 'ho-chi-minh-city', 'da-nang', 'gia-lai', 'cao-bang']
df_cities = df[df['queried_city'].isin(target_cities)].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(data=df_cities, x='queried_city', y='aqi',
            order=target_cities, ax=axes[0], palette='RdYlGn_r')
axes[0].set_title('AQI Distribution by City (Boxplot)')
axes[0].set_xlabel('')
axes[0].axhline(100, color='orange', linestyle='--', alpha=0.7, label='Moderate (100)')
axes[0].axhline(150, color='red',    linestyle='--', alpha=0.7, label='Unhealthy (150)')
axes[0].legend()

for city in target_cities:
    s = df_cities.loc[df_cities['queried_city'] == city, 'aqi'].dropna()
    if len(s) > 0:
        axes[1].hist(s, bins=40, alpha=0.5, label=city, density=True)
axes[1].set_title('AQI Density by City')
axes[1].set_xlabel('AQI'); axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
summary = df_cities.groupby('queried_city')['aqi'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max'])
summary['pct_unhealthy'] = df_cities.assign(un=df_cities['aqi'] > 100).groupby('queried_city')['un'].mean().mul(100).round(1)
summary.round(2).sort_values('mean', ascending=False)

**Insight nhanh**: Ha Noi và Cao Bang có median AQI cao hơn đáng kể so với Da Nang/HCMC — xu hướng này khớp với các nghiên cứu về ô nhiễm miền Bắc Việt Nam (PM2.5 mùa đông cao).

## 4. Correlation Analysis

In [ ]:
corr_cols = ['aqi', 'pm25', 'pm10', 'co', 'no2', 'o3', 'so2',
             'humidity', 'temperature', 'pressure', 'wind']
corr = df_cities[corr_cols].corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Spearman Correlation — AQI vs Pollutants & Weather')
plt.tight_layout(); plt.show()

aqi_corr = corr['aqi'].drop('aqi').sort_values(key=abs, ascending=False)
print('Top features correlated with AQI:')
print(aqi_corr.to_string())

**Insight**: AQI có correlation cao nhất với PM2.5 (đúng như kỳ vọng vì AQI ở VN thường bị dominated by PM2.5). Temperature/humidity có correlation âm yếu — mùa đông (thấp T, cao H) → AQI cao hơn.

## 5. Temporal Patterns

In [ ]:
df_t = df_cities.dropna(subset=['measured_at']).copy()
df_t['hour']  = df_t['measured_at'].dt.hour
df_t['dow']   = df_t['measured_at'].dt.dayofweek
df_t['month'] = df_t['measured_at'].dt.month
df_t['date']  = df_t['measured_at'].dt.date

fig, axes = plt.subplots(2, 2, figsize=(15, 9))

hourly = df_t.groupby(['queried_city', 'hour'])['aqi'].mean().reset_index()
for city in target_cities:
    d = hourly[hourly['queried_city'] == city]
    if len(d) > 0:
        axes[0, 0].plot(d['hour'], d['aqi'], marker='o', label=city)
axes[0, 0].set_title('Average AQI by Hour of Day')
axes[0, 0].set_xlabel('Hour'); axes[0, 0].set_ylabel('Mean AQI')
axes[0, 0].legend()

dow_map = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dowly = df_t.groupby(['queried_city', 'dow'])['aqi'].mean().reset_index()
pivot_dow = dowly.pivot(index='dow', columns='queried_city', values='aqi')
pivot_dow.index = [dow_map[i] for i in pivot_dow.index]
pivot_dow.plot(ax=axes[0, 1], marker='o')
axes[0, 1].set_title('Average AQI by Day of Week')
axes[0, 1].set_ylabel('Mean AQI')

monthly = df_t.groupby(['queried_city', 'month'])['aqi'].mean().reset_index()
pivot_m = monthly.pivot(index='month', columns='queried_city', values='aqi')
pivot_m.plot(ax=axes[1, 0], marker='o')
axes[1, 0].set_title('Average AQI by Month (Seasonality)')
axes[1, 0].set_ylabel('Mean AQI')
axes[1, 0].set_xticks(range(1, 13))

daily = df_t.groupby(['queried_city', 'date'])['aqi'].mean().reset_index()
for city in target_cities:
    d = daily[daily['queried_city'] == city].sort_values('date')
    if len(d) > 0:
        axes[1, 1].plot(d['date'], d['aqi'], label=city, alpha=0.7)
axes[1, 1].set_title('Daily AQI Time Series 2021')
axes[1, 1].set_ylabel('Mean AQI')
axes[1, 1].legend()
plt.setp(axes[1, 1].xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout(); plt.show()

**Insights**:
- **Hourly**: AQI tăng sáng sớm (~7-9h, traffic) và tối (~20-22h, inversion layer)
- **Monthly**: Mùa đông (tháng 11-2) cao nhất, mùa hè (tháng 6-8) thấp nhất — do gió mùa + inversion
- **Daily**: Ha Noi có spikes rất rõ vào mùa đông 2021

## 6. Seasonal Decomposition (STL)

In [ ]:
from statsmodels.tsa.seasonal import STL

hn = (df_t[df_t['queried_city'] == 'ha-noi']
      .groupby('date')['aqi'].mean()
      .rename_axis('date').to_frame('aqi'))
hn.index = pd.to_datetime(hn.index)
hn = hn.asfreq('D').interpolate(method='linear', limit=5)
hn = hn.dropna()
print(f'Ha Noi daily series: {len(hn)} points, {hn.index.min()} → {hn.index.max()}')

if len(hn) >= 30:
    stl = STL(hn['aqi'], period=7, robust=True).fit()
    fig = stl.plot(); fig.set_size_inches(14, 8)
    fig.suptitle('STL Decomposition — Ha Noi Daily AQI', y=1.02)
    plt.tight_layout(); plt.show()
else:
    print('Not enough data points for STL decomposition')

## 7. Dominant Pollutant Analysis

In [ ]:
dp = df_cities.groupby(['queried_city', 'dominant_pollutant']).size().reset_index(name='count')
dp_pivot = dp.pivot(index='queried_city', columns='dominant_pollutant', values='count').fillna(0)
dp_pct = dp_pivot.div(dp_pivot.sum(axis=1), axis=0).mul(100)

fig, ax = plt.subplots(figsize=(11, 5))
dp_pct.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
ax.set_title('Dominant Pollutant Share (%) by City')
ax.set_ylabel('% of records')
ax.set_xlabel('')
ax.legend(title='Pollutant', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

print('PM2.5 dominance (%):')
print(dp_pct.get('pm25', pd.Series()).round(1).to_string())

## 8. Geospatial Overview

In [ ]:
station_map = (df.dropna(subset=['lat', 'lon', 'aqi'])
                 .groupby(['station_id', 'station_name', 'lat', 'lon'], as_index=False)
                 ['aqi'].mean())
station_map = station_map[
    (station_map['lat'].between(8, 24)) & (station_map['lon'].between(102, 110))
]
print(f'Stations on map: {len(station_map)}')

fig, ax = plt.subplots(figsize=(10, 12))
sc = ax.scatter(station_map['lon'], station_map['lat'],
                c=station_map['aqi'], cmap='RdYlGn_r',
                s=60, edgecolor='black', linewidth=0.3,
                vmin=0, vmax=station_map['aqi'].quantile(0.95))
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Vietnam Monitoring Stations — Mean AQI 2021')
cbar = plt.colorbar(sc, ax=ax, label='Mean AQI')
plt.tight_layout(); plt.show()

## 9. Key Takeaways cho Modeling

Sau EDA, các quyết định modeling quan trọng:

| Observation | Implication cho Model |
|---|---|
| AQI có **strong seasonality** theo tháng và giờ | Cần `hour`, `month`, lag features |
| AQI có **spike events** (mùa đông Hà Nội) | Anomaly detection relevant, cần robust loss |
| PM2.5 correlation với AQI gần 1 | Có thể dùng làm feature hoặc target proxy |
| Missing values **không uniform** | Cần per-column imputation strategy |
| Weather correlation yếu nhưng có | Keep as features (non-linear via XGBoost) |
| AQI mỗi city có distribution khác biệt | Train per-city hoặc thêm city embedding |

**Next**: `02_feature_engineering.ipynb` — build lag, rolling, time features